# 03 - Assistant GenAI : text-to-SQL + narrateur

Pose une question en langage naturel : l'assistant génère une requête SQL,
l'exécute sur la base, et rédige une réponse pour un interlocuteur métier.
Garde-fous : SELECT uniquement. Nécessite `data/openpayments.sqlite` et
l'application **Ollama** (llama3.2) en marche.

In [ ]:
import os, re, sqlite3, pathlib
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

os.environ.setdefault("LLM_PROVIDER", "ollama")
os.environ.setdefault("OLLAMA_MODEL", "llama3.2")

ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT / "data").exists() else ROOT.parent
DB_PATH = ROOT / "data" / "openpayments.sqlite"

SCHEMA = """
Table `paiements` (un paiement d'un laboratoire a un professionnel de sante) :
- professionnel_id (TEXT) : identifiant du professionnel de sante
- etat (TEXT)             : etat (ex. 'WV')
- specialite (TEXT)       : specialite du professionnel
- laboratoire (TEXT)      : nom du laboratoire payeur
- montant (REAL)          : montant du paiement en USD
- nature (TEXT)           : nature ('Food and Beverage', 'Travel and Lodging', 'Consulting Fee', ...)
- annee (INTEGER)         : 2022 ou 2023
"""

SQL_PROMPT = (
    "Tu traduis une question en une requete SQL SQLite.\n{schema}\n"
    "Regles STRICTES : uniquement un SELECT ; une seule instruction sans point-virgule ;\n"
    "table `paiements` uniquement ; LIMIT 20 max si liste ; reponds avec la requete SEULE.\n\n"
    "Question : {question}\nSQL :"
)
NARRATE_PROMPT = (
    "Tu es analyste de donnees commerciales pharma. En 1 a 3 phrases claires,\n"
    "reponds a la question a partir du resultat SQL.\n\n"
    "Question : {question}\nResultat :\n{result}\n\nReponse :"
)

## Vue sémantique + garde-fous + chaîne

In [ ]:
def get_llm(temperature=0.0):
    from langchain_ollama import ChatOllama
    return ChatOllama(model=os.getenv("OLLAMA_MODEL", "llama3.2"), temperature=temperature)

def ensure_view():
    with sqlite3.connect(DB_PATH) as con:
        con.execute("""
        CREATE VIEW IF NOT EXISTS paiements AS
        SELECT covered_recipient_profile_id AS professionnel_id, recipient_state AS etat,
               specialty AS specialite,
               applicable_manufacturer_or_applicable_gpo_making_payment_name AS laboratoire,
               total_amount_of_payment_usdollars AS montant,
               nature_of_payment_or_transfer_of_value AS nature, program_year AS annee
        FROM payments""")

def extract_sql(text):
    text = re.sub(r"```(?:sql)?", "", text, flags=re.IGNORECASE).strip("` \n")
    m = re.search(r"(?is)\bselect\b.*", text)
    return (m.group(0) if m else text).strip().rstrip(";").strip()

def is_safe(sql):
    low = " " + sql.lower().strip() + " "
    if not low.strip().startswith("select") or ";" in sql:
        return False
    return not any(k in low for k in ("drop ", "delete ", "update ", "insert ", "alter ", "pragma"))

def run_sql(sql):
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql(sql, con)

def ask(question, max_retries=1, verbose=True):
    ensure_view()
    llm = get_llm()
    sql_chain = ChatPromptTemplate.from_template(SQL_PROMPT) | llm | StrOutputParser()
    error, sql, df = None, "", None
    for _ in range(max_retries + 1):
        q = question if error is None else f"{question}\n(Requete precedente en echec: {error}. Corrige.)"
        sql = extract_sql(sql_chain.invoke({"schema": SCHEMA, "question": q}))
        if not is_safe(sql):
            return "Requete refusee (SELECT uniquement).", sql, None
        try:
            df = run_sql(sql); error = None; break
        except Exception as exc:
            error = str(exc)
    if error is not None:
        return f"Echec SQL : {error}", sql, None
    narrate = ChatPromptTemplate.from_template(NARRATE_PROMPT) | llm | StrOutputParser()
    answer = narrate.invoke({"question": question, "result": df.head(20).to_string(index=False)})
    if verbose:
        print("SQL:", sql, "\n"); print(df.head(10).to_string(index=False))
    return answer, sql, df

## Démonstration

In [ ]:
answer, sql, df = ask("Combien de professionnels de sante distincts en 2023 ?")
print("\nREPONSE:", answer)

In [ ]:
answer, sql, df = ask("Quels sont les 5 laboratoires qui depensent le plus au total ?")
print("\nREPONSE:", answer)